# Augmentation Ablation: Effect on In-Domain and Out-of-Domain Performance

Experiment E3 (`docs/experiments/cross_domain_experiments_details.md`): Tissue-Region-Net on GSDB
(Prov-GigaPath embeddings, binary inflammation), trained on scanner-1 H&E. All runs share seed, splits and
evaluation protocol and differ only in the augmentation setup.

| Group | Runs | Question |
|---|---|---|
| **Ladder** | none → +GN → +FD → +MU → +SN → +MS (full) | What does each augmentation add on top of the previous ones? |
| **Leave-one-out (LOO)** | full minus one component | How much does the full set lose without each component? |
| **Consistency** | λ = 0, 0.225, 0.45 (full) | Effect of the augmentation-consistency loss weight |
| **Graph level** | {Plain, Plain + IN, Ref} config × {non-augmented, augmented} graph corpus | Augmentation at graph-build time vs. training time, and the effect of graph instance norm |

Abbreviations: GN feature Gaussian noise, FD feature dropout, MU graph Mixup, SN feature scale noise,
MS graph MixStyle, λ augmentation-consistency weight, IN graph instance norm (node features standardised per
slide graph), G-aug / G-noaug augmented / non-augmented graph corpus. Plain = no training augmentation, no
consistency loss, no IN (the E1 graph model); Plain + IN = Plain with IN; Ref = full augmentation set, λ = 0.45, IN.
All training-level runs (ladder, LOO, consistency) use IN.
The design matrix in Figure 1 is read from each run's training meta file, not typed in by hand.

**Protocol.** Checkpoint (max. validation slide F2) and decision threshold come from the scanner-1 H&E
validation split. ID test = `test` (scanner 1, H&E/MG/PAS, 53 slides), OOD test = `test_scanner2`
(scanner 2, H&E/MG/PAS, 90 slides, patient-disjoint). No recalibration on either test set.

ΔAUC = AUC(OOD) − AUC(ID) = −Gap, where the generalization gap used in the docs and the thesis is
Gap = AUC(S1) − AUC(S2); a ΔAUC of −5.6 pp therefore corresponds to Gap = 0.056.

**Statistics.** 95 % percentile CIs from a patient-level cluster bootstrap (B = `N_BOOT`). ΔAUC (OOD − ID)
uses independent bootstraps of the two patient-disjoint sets. Comparisons with the reference run (full
augmentation set) are paired on the same slides; two-sided bootstrap *p*-values, Holm-corrected. Validation
AUCs (39 usable slides, no CI) are close to 100 % for almost all runs and therefore barely separate the
configurations — the test sets are needed for that.

**Missing evaluations** are shown as gaps; Section 1 prints the `evaluate_model.py` commands that fill them.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.lines import Line2D
from sklearn.metrics import roc_auc_score

REPO_ROOT = Path("/home/stud/hweber/master_code")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.plot_style import IEEE_COLORS, apply_ieee_style, figsize

META_DIR = REPO_ROOT / "results/metas/training_metas"
SPLITS_DIR = REPO_ROOT / "data/splits/gsdb/seed_42"
OUT_DIR = REPO_ROOT / "notebooks/results/augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
# Unique runs: {run id: {"ID": evaluation path, "OOD": evaluation path, "meta": training meta}}
# Paths are relative to the repository root or absolute. An evaluation path may point to the evaluation
# folder (containing split_<split>/) or directly to the split_<split> folder. None = not evaluated.
# ID and OOD must come from the same checkpoint -- the notebook warns if their validation thresholds differ.
# "meta" defines the augmentation design matrix and is used to print the command for missing evaluations.
RUNS = {
    "none": {
        "ID":   "results/evaluations/e3-train-aug00-none_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_153022",
        "OOD":  "results/test_results/e3-train-aug00-none_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_trainheonly_None_tissue_region_net_20261017_153039",
        "meta": "results/metas/training_metas/e3-train-aug00-none_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_dataset-gsdb_model_type-gn_8e851a3d_mil_cached_training_method-function_20261012_160503.yaml",
    },
    "gn": {
        "ID":   "results/evaluations/e3-train-ladder1-gaussnoise_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_01_plus_feature_gaussian_noise_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_170548",
        "OOD":  "results/test_results/e3-train-ladder1-gaussnoise_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_01_plus_feature_gaussian_noise_trainheonly_None_tissue_region_net_20261017_154735",
        "meta": "results/metas/training_metas/e3-train-ladder1-gaussnoise_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_01_plus_feature_gaussian_n_44452e1d_mil_cached_training_method-function_20261012_175922.yaml",
    },
    "gn_fd": {
        "ID":   "results/evaluations/e3-train-ladder2-dropout_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_02_plus_feature_dropout_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_201326",
        "OOD":  "results/test_results/e3-train-ladder2-dropout_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_02_plus_feature_dropout_trainheonly_None_tissue_region_net_20261017_161323",
        "meta": "results/metas/training_metas/e3-train-ladder2-dropout_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_02_plus_feature_dropout_datas_27d069c4_mil_cached_training_method-function_20261012_211051.yaml",
    },
    "gn_fd_mu": {
        "ID":   "results/evaluations/e3-train-ladder3-mixup_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_03_plus_graph_mixup_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_201452",
        "OOD":  "results/test_results/e3-train-ladder3-mixup_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_03_plus_graph_mixup_trainheonly_None_tissue_region_net_20261017_161700",
        "meta": "results/metas/training_metas/e3-train-ladder3-mixup_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_03_plus_graph_mixup_dataset-gsd_c5b62c54_mil_cached_training_method-function_20261012_211832.yaml",
    },
    "gn_fd_mu_sn": {   # = LOO without MS
        "ID":   "results/evaluations/e3-train-ladder4-scalenoise_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_04_plus_feature_scale_noise_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_214225",
        "OOD":  "results/test_results/e3-train-ladder4-scalenoise_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_04_plus_feature_scale_noise_trainheonly_None_tissue_region_net_20261017_164015",
        "meta": "results/metas/training_metas/e3-train-ladder4-scalenoise_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_04_plus_feature_scale_nois_7eadbe2b_mil_cached_training_method-function_20261012_224651.yaml",
    },
    "full": {   # full set (= augablation_05_full = consistablation_02_full)
        "ID":   "results/evaluations/e2-gsdb-bin-gigapath_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_080503",
        "OOD":  "results/test_results/e2-gsdb-bin-gigapath_inflammation_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_trainheonly_None_tissue_region_net_20261017_150610",
        "meta": "results/metas/training_metas/e2-gsdb-bin-gigapath_task-inflammation_heonly_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-gsdb_model_4a91c1b7_mil_cached_training_method-function_20261012_090310.yaml",
    },
    "loo_gn": {
        "ID":   "results/evaluations/e3-train-loo-no-gaussnoise_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_gaussian_noise_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_214324",
        "OOD":  "results/test_results/e3-train-loo-no-gaussnoise_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_gaussian_noise_trainheonly_None_tissue_region_net_20261017_170501",
        "meta": "results/metas/training_metas/e3-train-loo-no-gaussnoise_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_gaussian_c966f727_mil_cached_training_method-function_20261012_223149.yaml",
    },
    "loo_fd": {
        "ID":   "results/evaluations/e3-train-loo-no-dropout_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_dropout_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_230052",
        "OOD":  "results/test_results/e3-train-loo-no-dropout_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_dropout_trainheonly_None_tissue_region_net_20261017_170048",
        "meta": "results/metas/training_metas/e3-train-loo-no-dropout_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_dropout_data_f5202c35_mil_cached_training_method-function_20261012_233444.yaml",
    },
    "loo_mu": {
        "ID":   "results/evaluations/e3-train-loo-no-mixup_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_graph_mixup_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_230220",
        "OOD":  "results/test_results/e3-train-loo-no-mixup_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_graph_mixup_trainheonly_None_tissue_region_net_20261017_170957",
        "meta": "results/metas/training_metas/e3-train-loo-no-mixup_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_graph_mixup_dataset-gs_c34a2102_mil_cached_training_method-function_20261012_235439.yaml",
    },
    "loo_sn": {
        "ID":   "results/evaluations/e3-train-loo-no-scalenoise_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_scale_noise_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_234019",
        "OOD":  "results/test_results/e3-train-loo-no-scalenoise_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_scale_noise_trainheonly_None_tissue_region_net_20261017_171406",
        "meta": "results/metas/training_metas/e3-train-loo-no-scalenoise_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_loo_minus_feature_scale_noi_cda85172_mil_cached_training_method-function_20261013_004034.yaml",
    },
    "cons_0": {
        "ID":   "results/evaluations/e3-train-consist-off_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_00_off_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261013_000105",
        "OOD":  "results/test_results/e3-train-consist-off_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_00_off_trainheonly_None_tissue_region_net_20261017_154033",
        "meta": "results/metas/training_metas/e3-train-consist-off_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_00_off_dataset-gsdb_model_typ_58556fb5_mil_cached_training_method-function_20261013_003916.yaml",
    },
    "cons_half": {
        "ID":   "results/evaluations/e3-train-consist-half_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_01_half_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261013_085616",
        "OOD":  "results/test_results/e3-train-consist-half_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_01_half_trainheonly_None_tissue_region_net_20261017_153633",
        "meta": "results/metas/training_metas/e3-train-consist-half_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_consistablation_01_half_dataset-gsdb_model_t_9a58d322_mil_cached_training_method-function_20261013_094834.yaml",
    },
    "plain_noaug": {
        "ID":   "results/evaluations/e1-graph-plain-noaug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261020_093028",
        "OOD":  "results/test_results/e1-graph-plain-noaug_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_trainheonly_None_tissue_region_net_20261020_100658",
        "meta": "results/metas/training_metas/e1-graph-plain-noaug_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_dataset-gsdb_m_d5e38cd6_mil_cached_training_method-function_20261020_095839.yaml",
    },
    "plain_aug": {
        "ID":   "results/evaluations/e3-graph-plain-aug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261020_144013",
        "OOD":  "results/test_results/e3-graph-plain-aug_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_trainheonly_None_tissue_region_net_20261020_162032",
        "meta": "results/metas/training_metas/e3-graph-plain-aug_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_dataset-gsdb_mod_e65c3fed_mil_cached_training_method-function_20261020_150123.yaml",
    },
    "plain_in_noaug": {
        "ID":   "results/evaluations/e3-graph-plain-in-noaug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261020_150517",
        "OOD":  "results/test_results/e3-graph-plain-in-noaug_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_trainheonly_None_tissue_region_net_20261020_163700",
        "meta": "results/metas/training_metas/e3-graph-plain-in-noaug_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_da_aa7dceb5_mil_cached_training_method-function_20261020_153042.yaml",
    },
    "plain_in_aug": {
        "ID":   "results/evaluations/e3-graph-plain-in-aug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261020_153630",
        "OOD":  "results/test_results/e3-graph-plain-in-aug_inflammation_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_trainheonly_None_tissue_region_net_20261020_163013",
        "meta": "results/metas/training_metas/e3-graph-plain-in-aug_task-inflammation_heonly_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_instnorm_data_8bc429d8_mil_cached_training_method-function_20261020_155726.yaml",
    },
    "ref_noaug": {
        "ID":   "results/evaluations/e3-graph-ref-noaug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_131000",
        "OOD":  "results/test_results/e3-graph-ref-noaug_inflammation_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_trainheonly_None_tissue_region_net_20261017_152547",
        "meta": "results/metas/training_metas/e3-graph-ref-noaug_task-inflammation_heonly_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-gsdb_model_ty_2c102f6d_mil_cached_training_method-function_20261012_140853.yaml",
    },
}
# Groups: ordered (run id, display label). A run may appear in several groups.
GROUPS = {
    "Ladder":      [("none", "none"), ("gn", "+GN"), ("gn_fd", "+FD"), ("gn_fd_mu", "+MU"),
                    ("gn_fd_mu_sn", "+SN"), ("full", "+MS (full)")],
    "LOO":         [("loo_gn", "−GN"), ("loo_fd", "−FD"), ("loo_mu", "−MU"), ("loo_sn", "−SN"),
                    ("gn_fd_mu_sn", "−MS")],
    "Consistency": [("cons_0", "λ = 0"), ("cons_half", "λ = 0.225"), ("full", "λ = 0.45")],
    "Graph level": [("plain_noaug", "Plain · G-noaug"), ("plain_aug", "Plain · G-aug"),
                    ("plain_in_noaug", "Plain+IN · G-noaug"), ("plain_in_aug", "Plain+IN · G-aug"),
                    ("ref_noaug", "Ref · G-noaug"), ("full", "Ref · G-aug")],
}
REFERENCE = "full"
DOMAINS = {"ID": "test", "OOD": "test_scanner2"}
DOMAIN_LONG = {"VAL": "Validation (S1 · H&E)", "ID": "ID test (S1)", "OOD": "OOD test (S2)"}
AUG_COLUMNS = [("feature_gaussian_noise", "GN"), ("feature_dropout", "FD"), ("graph_mixup", "MU"),
               ("feature_scale_noise", "SN"), ("graph_mixstyle", "MS")]
N_BOOT = 2000
SEED = 0
SAVE_PREFIX = "e3"

## 1 · Data

In [ ]:
def _one(directory, pattern):
    hits = sorted(directory.glob(pattern)) if directory.is_dir() else []
    return hits[0] if hits else None


def resolve_path(path):
    # Repository-relative or absolute path; None stays None. A given path that does not exist is an error.
    if path is None:
        return None
    p = Path(path)
    p = p if p.is_absolute() else REPO_ROOT / p
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p


def resolve_split_dir(path, split):
    # Accepts an evaluation folder (containing split_<split>/) or the split_<split> folder itself.
    p = resolve_path(path)
    if p is None:
        return None
    split_dir = p if p.name == f"split_{split}" else p / f"split_{split}"
    if not _one(split_dir, f"*_{split}_slide_metrics.json"):
        raise FileNotFoundError(f"No *_{split}_slide_metrics.json in {split_dir}")
    return split_dir


def find_key(d, key):
    if isinstance(d, dict):
        if key in d:
            return d[key]
        for v in d.values():
            hit = find_key(v, key)
            if hit is not None:
                return hit
    return None


def load_split(split_dir, split):
    pred_file = _one(split_dir, f"*_{split}_slide_predictions.csv") or _one(split_dir, f"*_{split}_predictions.csv")
    preds = pd.read_csv(pred_file)[["slide_name", "prob", "label"]]
    if preds["slide_name"].duplicated().any():
        raise ValueError(f"{pred_file} is not slide-level")
    metrics = json.load(open(_one(split_dir, f"*_{split}_slide_metrics.json")))
    at_val = metrics.get("at_validation_threshold") or metrics.get("slide_at_validation_threshold")
    val_file = split_dir.parent / "validation_thresholds" / "validation_thresholds.json"
    val_auc = json.load(open(val_file)).get("validation_slide_auroc") if val_file.exists() else None
    return {"preds": preds, "at_val": at_val, "val_auc": val_auc}


cohorts = {}
for dom, split in DOMAINS.items():
    c = pd.read_csv(SPLITS_DIR / f"{split}.csv")
    c = c[c["inflammation_status"].isin(["inflamed", "noninflamed"])].reset_index(drop=True)
    c["label"] = (c["inflammation_status"] == "inflamed").astype(int)
    cohorts[dom] = c[["slide_name", "patient_id", "stain", "label"]]

# Design matrix from the training metas
design, metas = {}, {}
for rid, paths in RUNS.items():
    meta = resolve_path(paths.get("meta"))
    metas[rid] = meta
    if meta is None:
        design[rid] = None
        continue
    m = yaml.safe_load(open(meta))
    augs = [a["name"] for a in (find_key(m, "train_augmentations") or [])]
    corpus = str(find_key(m, "graph_target_dir") or "")
    design[rid] = {**{short: name in augs for name, short in AUG_COLUMNS},
                   "λ": float(find_key(m, "augmentation_consistency_weight") or 0.0),
                   "IN": bool(find_key(m, "graph_instance_norm")),
                   "G-aug": not corpus.rstrip("/").endswith("_noaug")}

results, missing = {}, []
for rid, paths in RUNS.items():
    val_auc = None
    for dom, split in DOMAINS.items():
        split_dir = resolve_split_dir(paths.get(dom), split)
        if split_dir is None:
            missing.append((rid, split))
            continue
        r = load_split(split_dir, split)
        merged = cohorts[dom].merge(r["preds"], on="slide_name", how="left", suffixes=("", "_pred"))
        known = merged["prob"].notna()
        if (merged.loc[known, "label"] != merged.loc[known, "label_pred"]).any():
            raise ValueError(f"Label mismatch: {rid} / {dom}")
        r["prob"] = merged["prob"].to_numpy()
        results[(rid, dom)] = r
        val_auc = val_auc if val_auc is not None else r["val_auc"]
    if val_auc is not None:
        results[(rid, "VAL")] = val_auc

avail = pd.DataFrame({rid: {DOMAIN_LONG[d]: ("✓" if (rid, d) in results else "missing") for d in DOMAINS}
                      for rid in RUNS}).T
avail.insert(0, "Run", pd.Series({rid: metas[rid].name.split("_task-")[0] if metas[rid] else "–" for rid in RUNS}))
avail

In [ ]:
# Commands that fill the missing evaluations (one evaluate_model.py call per run, all missing splits at once)
todo = {}
for rid, split in missing:
    todo.setdefault(rid, []).append(split)
if todo:
    print("# Missing evaluations -- run from the repository root:")
    for rid, splits in todo.items():
        meta = metas[rid]
        meta_arg = meta.relative_to(REPO_ROOT) if meta else f"<meta file of run {rid}>"
        print(f"PYTHONHASHSEED=42 python src/entrypoint/evaluate_model.py --splits {' '.join(splits)} \\\n"
              f"  --meta_file {meta_arg}")
else:
    print("All evaluations present.")

In [ ]:
# Protocol check: one validation threshold per run, stored confusion matrix reproduced.
def confusion(y, p, t):
    pred = (p > t).astype(int)
    return {"tp": int(((pred == 1) & (y == 1)).sum()), "fn": int(((pred == 0) & (y == 1)).sum()),
            "fp": int(((pred == 1) & (y == 0)).sum()), "tn": int(((pred == 0) & (y == 0)).sum())}


problems = 0
for rid in RUNS:
    thr = {d: results[(rid, d)]["at_val"]["threshold"] for d in DOMAINS if (rid, d) in results}
    if len(set(np.round(list(thr.values()), 9))) > 1:
        print(f"WARNING {rid}: thresholds differ between domains {thr}")
        problems += 1
    for dom in DOMAINS:
        if (rid, dom) in results:
            r, y = results[(rid, dom)], cohorts[dom]["label"].to_numpy()
            ok = ~np.isnan(r["prob"])
            cm = confusion(y[ok], r["prob"][ok], r["at_val"]["threshold"])
            if cm != {k: r["at_val"][k] for k in cm}:
                print(f"MISMATCH {rid} {dom}: {cm} vs stored")
                problems += 1
print("Protocol checks passed." if not problems else f"{problems} problem(s) found.")

## 2 · Bootstrap estimates

In [ ]:
rng = np.random.default_rng(SEED)


def cluster_bootstrap(patient_ids, n_boot, rng):
    patients, inverse = np.unique(patient_ids, return_inverse=True)
    rows_of = [np.flatnonzero(inverse == k) for k in range(len(patients))]
    return [np.concatenate([rows_of[k] for k in rng.integers(0, len(patients), len(patients))])
            for _ in range(n_boot)]


boot = {dom: cluster_bootstrap(c["patient_id"].to_numpy(), N_BOOT, rng) for dom, c in cohorts.items()}


def metrics(y, p, t):
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    cm = confusion(y, p, t)
    sens = cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else np.nan
    spec = cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else np.nan
    return {"AUC": roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan,
            "Sensitivity": sens, "Specificity": spec, "Balanced accuracy": np.nanmean([sens, spec])}


def ci(x):
    x = pd.Series(x).dropna()
    return np.percentile(x, [2.5, 97.5]) if len(x) else (np.nan, np.nan)


est = {}
for (rid, dom), r in results.items():
    if dom == "VAL":
        continue
    y, p, t = cohorts[dom]["label"].to_numpy(), r["prob"], r["at_val"]["threshold"]
    est[(rid, dom)] = (metrics(y, p, t), pd.DataFrame([metrics(y[i], p[i], t) for i in boot[dom]]))


def delta_auc(rid):
    if not all((rid, d) in est for d in ("ID", "OOD")):
        return None
    reps = est[(rid, "OOD")][1]["AUC"].to_numpy() - est[(rid, "ID")][1]["AUC"].to_numpy()
    return est[(rid, "OOD")][0]["AUC"] - est[(rid, "ID")][0]["AUC"], ci(reps)


def paired_delta(rid, dom, ref=REFERENCE):
    # AUC(run) − AUC(reference) on the slides both predicted, paired patient-level bootstrap.
    if (rid, dom) not in results or (ref, dom) not in results or rid == ref:
        return None
    y = cohorts[dom]["label"].to_numpy()
    p, q = results[(rid, dom)]["prob"], results[(ref, dom)]["prob"]
    common = ~np.isnan(p) & ~np.isnan(q)
    diffs = []
    for idx in boot[dom]:
        i = idx[common[idx]]
        if 0 < y[i].sum() < len(i):
            diffs.append(roc_auc_score(y[i], p[i]) - roc_auc_score(y[i], q[i]))
    diffs = np.asarray(diffs)
    d = roc_auc_score(y[common], p[common]) - roc_auc_score(y[common], q[common])
    pval = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return d, np.percentile(diffs, [2.5, 97.5]), pval


deltas = {rid: delta_auc(rid) for rid in RUNS}
paired = {(rid, dom): paired_delta(rid, dom) for rid in RUNS for dom in DOMAINS}
print(f"{N_BOOT} patient-level bootstrap resamples per domain; {len(est)} run × domain estimates.")

## 3 · Overview: design and results

Left: augmentation setup of every run (● = active). Right: slide-level results — validation AUC, ID and OOD
test AUC, the domain-shift change ΔAUC = OOD − ID, and the OOD operating point at the validation threshold.
Each run appears once; the full set is the reference (bold row).

In [ ]:
apply_ieee_style()
W, _ = figsize("double")
C_ID, C_OOD, C_VAL = IEEE_COLORS[0], IEEE_COLORS[1], "0.45"
GROUP_COLORS = {g: IEEE_COLORS[(i + 2) % len(IEEE_COLORS)] for i, g in enumerate(GROUPS)}
# Colour-blind safe redundancy: domains differ in marker, fill and line style; groups in marker shape.
DOMAIN_STYLE = {"VAL": dict(color=C_VAL, marker="D", face="white", ls=":"),
                "ID": dict(color=C_ID, marker="o", face="white", ls="--"),
                "OOD": dict(color=C_OOD, marker="s", face=C_OOD, ls="-")}
GROUP_MARKERS = {g: ["^", "v", "P", "X"][i % 4] for i, g in enumerate(GROUPS)}


def domain_handles(domains=("VAL", "ID", "OOD")):
    return [Line2D([], [], color=DOMAIN_STYLE[d]["color"], marker=DOMAIN_STYLE[d]["marker"],
                   markerfacecolor=DOMAIN_STYLE[d]["face"], markeredgecolor=DOMAIN_STYLE[d]["color"],
                   linestyle=DOMAIN_STYLE[d]["ls"], label=DOMAIN_LONG[d]) for d in domains]


def save(fig, name):
    if SAVE_PREFIX:
        for ext in ("png", "pdf"):
            fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_{name}.{ext}", bbox_inches="tight")
        print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_{name}.png")


def panel(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left")


def style_heatmap_axis(ax, ncols, nrows):
    ax.set_xticks(np.arange(-0.5, ncols), minor=True)
    ax.set_yticks(np.arange(-0.5, nrows), minor=True)
    ax.grid(False)
    ax.grid(which="minor", color="white", linewidth=1.5, linestyle="-", alpha=1)
    ax.tick_params(which="both", length=0)
    for s in ax.spines.values():
        s.set_visible(False)


# One row per unique run, in group order
rows, seen = [], set()
for g, members in GROUPS.items():
    for rid, label in members:
        if rid not in seen:
            seen.add(rid)
            rows.append((g, rid, label))
row_ids = [rid for _, rid, _ in rows]
row_labels = [f"{label}" for _, _, label in rows]

design_cols = [s for _, s in AUG_COLUMNS] + ["λ", "IN", "G-aug"]
res_cols = ["Val AUC", "ID AUC", "OOD AUC", "OOD sens.", "OOD spec.", "OOD bal. acc."]
res_tick = ["Val\nAUC", "ID\nAUC", "OOD\nAUC", "OOD\nsens.", "OOD\nspec.", "OOD\nBA"]


def res_value(rid, col):
    if col == "Val AUC":
        v = results.get((rid, "VAL"))
        return 100 * v if v is not None else np.nan
    dom, key = {"ID AUC": ("ID", "AUC"), "OOD AUC": ("OOD", "AUC"), "OOD sens.": ("OOD", "Sensitivity"),
                "OOD spec.": ("OOD", "Specificity"), "OOD bal. acc.": ("OOD", "Balanced accuracy")}[col]
    return 100 * est[(rid, dom)][0][key] if (rid, dom) in est else np.nan


R = np.array([[res_value(rid, c) for c in res_cols] for rid in row_ids])
D = np.array([[100 * deltas[rid][0] if deltas[rid] else np.nan] for rid in row_ids])

n = len(rows)
fig, (ax_d, ax_r, ax_delta) = plt.subplots(1, 3, figsize=(W, 0.2 * n + 1.5), sharey=True,
                                           gridspec_kw={"width_ratios": [len(design_cols), len(res_cols), 1.3]})
# (a) design matrix
for i, rid in enumerate(row_ids):
    d = design[rid]
    for j, col in enumerate(design_cols):
        if d is None:
            ax_d.text(j, i, "?", ha="center", va="center", fontsize=6.5, color="0.5")
        elif col == "λ":
            ax_d.text(j, i, f"{d['λ']:g}", ha="center", va="center", fontsize=6.5, color="0.1" if d["λ"] else "0.6")
        else:
            on = d[col]
            ax_d.plot(j, i, "o", markersize=5 if on else 3, color="0.15" if on else "0.8",
                      markerfacecolor="0.15" if on else "white")
ax_d.set_xlim(-0.5, len(design_cols) - 0.5)
ax_d.set_xticks(range(len(design_cols)), design_cols)
ax_d.xaxis.tick_top()
ax_d.set_yticks(range(n), row_labels)
style_heatmap_axis(ax_d, len(design_cols), n)
ax_d.set_title("(a) Augmentation setup", loc="left", pad=22)

# (b) results heatmap
cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("0.93")
ax_r.imshow(np.ma.masked_invalid(R), cmap=cmap, vmin=40, vmax=100, aspect="auto")
for i in range(n):
    for j in range(len(res_cols)):
        v = R[i, j]
        ax_r.text(j, i, "–" if np.isnan(v) else f"{v:.0f}", ha="center", va="center", fontsize=6.5,
                  color="0.5" if np.isnan(v) else ("white" if v > 80 else "0.1"))
ax_r.set_xticks(range(len(res_cols)), res_tick, fontsize=7)
ax_r.xaxis.tick_top()
style_heatmap_axis(ax_r, len(res_cols), n)
ax_r.set_title("(b) Results (%)", loc="left", pad=22)

# (c) ΔAUC
dmax = max(10, 5 * np.ceil(np.nanmax(np.abs(D)) / 5)) if np.isfinite(D).any() else 10
cmap_d = plt.get_cmap("RdBu").copy()
cmap_d.set_bad("0.93")
ax_delta.imshow(np.ma.masked_invalid(D), cmap=cmap_d, vmin=-dmax, vmax=dmax, aspect="auto")
for i in range(n):
    v = D[i, 0]
    ax_delta.text(0, i, "–" if np.isnan(v) else f"{v:+.1f}", ha="center", va="center", fontsize=6.5,
                  color="0.5" if np.isnan(v) else ("white" if abs(v) > 0.55 * dmax else "0.1"))
ax_delta.set_xticks([0], ["ΔAUC\nOOD−ID"], fontsize=7)
ax_delta.xaxis.tick_top()
style_heatmap_axis(ax_delta, 1, n)
ax_delta.set_title("(c)", loc="left", pad=22)

# group separators, labels and reference highlight
boundaries, start = [], 0
for g in GROUPS:
    k = sum(1 for gg, _, _ in rows if gg == g)
    if k:
        boundaries.append((g, start, start + k - 1))
        start += k
# group labels sit left of the longest row label (measured), so they never overlap it
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
label_w = max(t.get_window_extent(renderer).width for t in ax_d.get_yticklabels()) * 72 / fig.dpi
for g, a, b in boundaries:
    for ax in (ax_d, ax_r, ax_delta):
        if a:
            ax.axhline(a - 0.5, color="0.35", linewidth=0.9)
    ax_d.annotate(g, xy=(0, (a + b) / 2), xycoords=("axes fraction", "data"), xytext=(-(label_w + 12), 0),
                  textcoords="offset points", ha="right", va="center", fontsize=7.5, rotation=0,
                  color=GROUP_COLORS[g], fontweight="bold")
for lab in ax_d.get_yticklabels():
    if row_ids[row_labels.index(lab.get_text())] == REFERENCE:
        lab.set_fontweight("bold")
ax_d.set_ylim(n - 0.5, -0.5)
fig.text(0.01, 0.022, "(a) ● = component active, ○ = inactive; λ = consistency weight; IN = graph instance norm; G-aug = augmented graph corpus.   "
         "(b) blue shade = score (%, 40–100); BA = balanced accuracy; OOD metrics at the validation threshold.",
         fontsize=6.5, color="0.35", ha="left", va="bottom")
fig.text(0.01, 0.0, "(c) red = AUC loss, blue = gain from ID to OOD (pp).   – = not evaluated.   "
         "Each run is shown once; the full set (bold) is also λ = 0.45 and Ref · G-aug.",
         fontsize=6.5, color="0.35", ha="left", va="bottom")
fig.tight_layout(rect=(0, 0.05, 1, 1))
save(fig, "aug_overview")
plt.show()

## 4 · Effect of each augmentation factor

Top row: slide-level AUC on validation (◇), ID test (○) and OOD test (●) with 95 % CIs.
Bottom row: ΔAUC = OOD − ID with 95 % CI; values closer to 0 mean less loss under the domain shift.
(a) cumulative ladder, (b) consistency weight, (c) graph-level 3 × 2 design (Plain → Plain + IN isolates the
instance norm, Plain + IN → Ref the training augmentation + consistency loss).

In [ ]:
effect_groups = ["Ladder", "Consistency", "Graph level"]
widths = [len(GROUPS[g]) for g in effect_groups]
fig, axes = plt.subplots(2, len(effect_groups), figsize=(W, 4.4), sharey="row",
                         gridspec_kw={"width_ratios": widths, "height_ratios": [1.5, 1]})
for col, (g, letter) in enumerate(zip(effect_groups, "abc")):
    ax, ax2 = axes[0, col], axes[1, col]
    members = GROUPS[g]
    x = np.arange(len(members))
    for dom, dx in (("VAL", -0.18), ("ID", 0.0), ("OOD", 0.18)):
        color, marker, face, ls = (DOMAIN_STYLE[dom][k] for k in ("color", "marker", "face", "ls"))
        xs, ys = [], []
        for k, (rid, _) in enumerate(members):
            have = (rid, "VAL") in results if dom == "VAL" else (rid, dom) in est
            if not have:
                xs.append(np.nan); ys.append(np.nan)   # breaks the line at missing evaluations
                continue
            if dom == "VAL":
                xs.append(k + dx); ys.append(100 * results[(rid, "VAL")])
                continue
            point, reps = est[(rid, dom)]
            lo, hi = 100 * ci(reps["AUC"])
            ax.plot([k + dx] * 2, [lo, hi], color=color, linewidth=1.1, alpha=0.75)
            xs.append(k + dx); ys.append(100 * point["AUC"])
        if np.isfinite(ys).any():
            ax.plot(xs, ys, linestyle=ls, color=color, linewidth=1.0, alpha=0.7)
            ax.plot(xs, ys, marker, color=color, markerfacecolor=face, markeredgecolor=color,
                    markeredgewidth=1.2, markersize=4 if marker == "D" else 5, linestyle="", zorder=4)
    for k, (rid, _) in enumerate(members):
        dd = deltas[rid]
        if dd is None:
            ax2.text(k, 0.92, "n/a", ha="center", va="top", fontsize=6.5, color="0.5",
                     transform=ax2.get_xaxis_transform())
            continue
        d, (lo, hi) = dd
        ax2.plot([k, k], [100 * lo, 100 * hi], color=GROUP_COLORS[g], linewidth=1.6)
        ax2.plot(k, 100 * d, GROUP_MARKERS[g], color=GROUP_COLORS[g], markersize=6, markeredgecolor="white", zorder=4)
    ax2.axhline(0, color="0.3", linestyle=":", linewidth=0.9)
    for a in (ax, ax2):
        a.set_xlim(-0.6, len(members) - 0.4)
        a.grid(axis="x", visible=False)
    ax.set_xticks(x, [""] * len(x))
    ax2.set_xticks(x, [lab for _, lab in members], rotation=35, ha="right")
    for k, (rid, _) in enumerate(members):
        if rid == REFERENCE:
            ax2.get_xticklabels()[k].set_fontweight("bold")
    panel(ax, letter, g)
axes[0, 0].set_ylabel("Slide-level AUC (%)")
axes[1, 0].set_ylabel("ΔAUC, OOD − ID (pp)")
fig.legend(handles=domain_handles() + [
    Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="", label="vertical bar: 95 % CI"),
    Line2D([], [], color="0.3", marker="^", linestyle="", label="bottom row: ΔAUC (marker = group)"),
    Line2D([], [], color="0.3", linestyle=":", label="no change (ΔAUC = 0)"),
    Line2D([], [], color="none", label="bold label = full set" + ("; n/a = OOD not evaluated"
                                       if any(deltas[r] is None for g in effect_groups for r, _ in GROUPS[g]) else ""))],
    loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.09, 1, 1))
save(fig, "aug_effects")
plt.show()

## 5 · Leave-one-out: contribution of each component

Change in AUC when one component is removed from the full set, ΔAUC = AUC(full − component) − AUC(full),
paired on the same slides. **Negative = the component helps** (removing it hurts); positive = the model is
better without it. Shown separately for the ID and the OOD test set.

In [ ]:
loo = GROUPS["LOO"]
fig, ax = plt.subplots(figsize=(W * 0.6, 0.42 * len(loo) + 1.1))
for k, (rid, label) in enumerate(loo):
    for dom, dy in (("ID", 0.14), ("OOD", -0.14)):
        color, marker, face = (DOMAIN_STYLE[dom][k] for k in ("color", "marker", "face"))
        pd_ = paired.get((rid, dom))
        if pd_ is None:
            continue
        d, (lo, hi), _ = pd_
        ax.plot([100 * lo, 100 * hi], [k + dy] * 2, color=color, linewidth=1.4, alpha=0.8)
        ax.plot(100 * d, k + dy, marker, markerfacecolor=face, markeredgecolor=color, markeredgewidth=1.3,
                markersize=6, zorder=4)
ax.axvline(0, color="0.3", linestyle=":", linewidth=0.9)
ax.set_yticks(range(len(loo)), [lab for _, lab in loo])
ax.set_ylim(len(loo) - 0.5, -0.5)
ax.set_xlabel("ΔAUC vs. full set (pp, paired 95 % CI)")
ax.grid(axis="y", visible=False)
ax.text(0.01, 1.01, "← component helps", transform=ax.transAxes, ha="left", va="bottom", fontsize=6.5, color="0.4")
ax.text(0.99, 1.01, "better without →", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5, color="0.4")
ax.legend(handles=domain_handles(("ID", "OOD")) + [
    Line2D([], [], color="0.4", linewidth=1.4, label="bar: paired 95 % CI"),
    Line2D([], [], color="0.3", linestyle=":", label="no difference to full set")],
    loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)
fig.tight_layout()
save(fig, "aug_loo")
plt.show()

## 6 · All runs ranked

Every run once, sorted by OOD AUC (runs without OOD evaluation at the bottom, sorted by ID AUC).
○ ID test, ● OOD test, bars = 95 % CI; the number on the right is ΔAUC = OOD − ID. Row labels are coloured by
group; the reference (full set) is bold.

In [ ]:
def label_of(rid):
    for g, members in GROUPS.items():
        for r, lab in members:
            if r == rid:
                return g, lab
    return None, rid


def sort_key(rid):
    if (rid, "OOD") in est:
        return (0, -est[(rid, "OOD")][0]["AUC"])
    if (rid, "ID") in est:
        return (1, -est[(rid, "ID")][0]["AUC"])
    return (2, 0)


ranked = sorted(RUNS, key=sort_key)
fig, ax = plt.subplots(figsize=(W * 0.72, 0.24 * len(ranked) + 1.0))
for k, rid in enumerate(ranked):
    pts = {}
    for dom, dy in (("ID", 0.12), ("OOD", -0.12)):
        color, marker, face = (DOMAIN_STYLE[dom][k] for k in ("color", "marker", "face"))
        if (rid, dom) not in est:
            continue
        point, reps = est[(rid, dom)]
        lo, hi = 100 * ci(reps["AUC"])
        ax.plot([lo, hi], [k - dy] * 2, color=color, linewidth=1.0, alpha=0.55)
        ax.plot(100 * point["AUC"], k - dy, marker, markerfacecolor=face, markeredgecolor=color,
                markeredgewidth=1.2, markersize=5, zorder=4)
        pts[dom] = 100 * point["AUC"]
    if len(pts) == 2:
        ax.plot([pts["ID"], pts["OOD"]], [k - 0.12, k + 0.12], color="0.6", linewidth=0.7, zorder=3)
    dd = deltas[rid]
    ax.text(1.01, k, f"{100 * dd[0]:+.1f}" if dd else "–", transform=ax.get_yaxis_transform(),
            ha="left", va="center", fontsize=6.5, color="0.15" if dd else "0.55")
labels = [f"{label_of(r)[1]}  [{label_of(r)[0]}]" for r in ranked]
ax.set_yticks(range(len(ranked)), labels, fontsize=7)
for tick, rid in zip(ax.get_yticklabels(), ranked):
    g = label_of(rid)[0]
    tick.set_color("0.1" if rid == REFERENCE else GROUP_COLORS.get(g, "0.2"))
    if rid == REFERENCE:
        tick.set_fontweight("bold")
ax.set_ylim(len(ranked) - 0.5, -0.5)
ax.set_xlabel("Slide-level AUC (%, 95 % CI)")
ax.grid(axis="y", visible=False)
ax.text(1.01, 1.0, "ΔAUC", transform=ax.transAxes, ha="left", va="bottom", fontsize=6.5, color="0.4")
ax.legend(handles=domain_handles(("ID", "OOD")) + [
    Line2D([], [], color="0.4", linewidth=1.0, alpha=0.6, label="bar: 95 % CI"),
    Line2D([], [], color="0.6", linewidth=0.7, label="grey link: same run ID ↔ OOD"),
    Line2D([], [], color="none", label="label colour = group [in brackets]; right: ΔAUC (pp)")],
    loc="upper center", bbox_to_anchor=(0.45, -0.06 - 1.2 / (0.24 * len(ranked) + 1.0) * 0.5), ncol=2, frameon=False)
fig.tight_layout()
save(fig, "aug_ranking")
plt.show()

**Operating points.** Sensitivity and false-positive rate (1 − specificity) at the validation threshold for
every run: ○ ID, ● OOD, arrow = shift under the domain change. Movement to the right means more false positives
on scanner 2; movement down means more missed inflamed slides.

In [ ]:
fig, ax = plt.subplots(figsize=(W * 0.62, W * 0.5))
done = set()
for g, members in GROUPS.items():
    for rid, _ in members:
        if rid in done:
            continue
        done.add(rid)
        c = GROUP_COLORS[g] if rid != REFERENCE else "0.1"
        for dom, face in (("ID", "white"), ("OOD", c)):
            if (rid, dom) in est:
                pt = est[(rid, dom)][0]
                mk = GROUP_MARKERS[g] if rid != REFERENCE else "*"
                ax.plot(100 * (1 - pt["Specificity"]), 100 * pt["Sensitivity"], mk, markerfacecolor=face,
                        markeredgecolor=c, markeredgewidth=1.1, markersize=8 if mk == "*" else 5.5,
                        zorder=6 if mk == "*" else 4)
        if all((rid, d) in est for d in ("ID", "OOD")):
            p0, p1 = est[(rid, "ID")][0], est[(rid, "OOD")][0]
            ax.annotate("", xy=(100 * (1 - p1["Specificity"]), 100 * p1["Sensitivity"]),
                        xytext=(100 * (1 - p0["Specificity"]), 100 * p0["Sensitivity"]),
                        arrowprops=dict(arrowstyle="-|>", color=c, lw=0.6, alpha=0.6, shrinkA=3, shrinkB=3))
ax.plot([0, 100], [0, 100], color="0.75", linestyle=":", linewidth=0.8)
xs_all = [100 * (1 - est[k][0]["Specificity"]) for k in est]
ys_all = [100 * est[k][0]["Sensitivity"] for k in est]
ax.set_xlim(-2, min(102, max(xs_all) + 8))
ax.set_ylim(max(-2, min(ys_all) - 8), 102)
ax.set_xlabel("1 − specificity (%)")
ax.set_ylabel("Sensitivity (%)")
handles = [Line2D([], [], marker=GROUP_MARKERS[g], color=GROUP_COLORS[g], linestyle="", label=g) for g in GROUPS]
handles += [Line2D([], [], marker="*", color="0.1", linestyle="", markersize=8, label="full set"),
            Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="", label="ID"),
            Line2D([], [], marker="o", color="0.3", linestyle="", label="OOD"),
            Line2D([], [], color="0.5", linewidth=0.6, marker=">", markersize=4, label="arrow: shift ID → OOD")]
if ax.get_ylim()[0] < ax.get_xlim()[1]:   # diagonal (chance level) lies inside the zoomed window
    handles.append(Line2D([], [], color="0.75", linestyle=":", label="chance level"))
fig.legend(handles=handles, loc="lower center", fontsize=6.5, ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.17, 1, 1))
save(fig, "aug_operating_points")
plt.show()

## 7 · Results tables

In [ ]:
def fmt_ci(point, reps, key):
    lo, hi = ci(reps[key])
    return f"{100 * point[key]:.1f} [{100 * lo:.1f}, {100 * hi:.1f}]"


def fmt_p(p):
    return f"< {1 / N_BOOT:.4f}".rstrip("0") if p < 1 / N_BOOT else f"{p:.3f}"


# Holm correction over all paired comparisons with the reference
comp = [(k, v) for k, v in paired.items() if v is not None]
pvals = np.array([v[2] for _, v in comp])
adj = np.empty_like(pvals)
running = 0.0
for rank, i in enumerate(np.argsort(pvals)):
    running = max(running, (len(pvals) - rank) * pvals[i])
    adj[i] = min(1.0, running)
holm = {k: a for (k, _), a in zip(comp, adj)}

rows = []
for g, members in GROUPS.items():
    for rid, label in members:
        row = {"Group": g, "Run": label,
               "Val AUC": f"{100 * results[(rid, 'VAL')]:.1f}" if (rid, "VAL") in results else "–"}
        for dom in DOMAINS:
            if (rid, dom) in est:
                point, reps = est[(rid, dom)]
                row[f"{dom} AUC"] = fmt_ci(point, reps, "AUC")
                row[f"{dom} sens. / spec."] = f"{100 * point['Sensitivity']:.0f} / {100 * point['Specificity']:.0f}"
            else:
                row[f"{dom} AUC"] = "–"
                row[f"{dom} sens. / spec."] = "–"
        dd = deltas[rid]
        row["ΔAUC (OOD − ID)"] = f"{100 * dd[0]:+.1f} [{100 * dd[1][0]:+.1f}, {100 * dd[1][1]:+.1f}]" if dd else "–"
        for dom in DOMAINS:
            pd_ = paired.get((rid, dom))
            row[f"{dom} vs. full (p Holm)"] = (f"{100 * pd_[0]:+.1f} ({fmt_p(holm[(rid, dom)])})"
                                               if pd_ is not None else ("ref." if rid == REFERENCE else "–"))
        rows.append(row)

table = pd.DataFrame(rows).set_index(["Group", "Run"])
table

In [ ]:
def nice(rid):
    return next((f"{lab} [{g}]" for g, mem in GROUPS.items() for r, lab in mem if r == rid), rid)


lines = []
ref_id, ref_ood = est.get((REFERENCE, "ID")), est.get((REFERENCE, "OOD"))
if ref_id:
    lines.append(f"Reference (full set): ID AUC {100 * ref_id[0]['AUC']:.1f}"
                 + (f", OOD AUC {100 * ref_ood[0]['AUC']:.1f}" if ref_ood else ", OOD not evaluated"))
for dom in DOMAINS:
    cands = [(rid, est[(rid, dom)][0]["AUC"]) for rid in RUNS if (rid, dom) in est]
    if cands:
        best = max(cands, key=lambda t: t[1])
        lines.append(f"Highest {dom} AUC: {nice(best[0])} ({100 * best[1]:.1f})")
sig = [(k, v) for k, v in paired.items() if v is not None and holm[k] < 0.05]
lines.append("Significant differences vs. full set (Holm < 0.05): "
             + ("; ".join(f"{nice(rid)} on {dom}: {100 * v[0]:+.1f} pp" for (rid, dom), v in sig) if sig else "no run"))
with_delta = [(rid, d[0]) for rid, d in deltas.items() if d]
if with_delta:
    best = max(with_delta, key=lambda t: t[1])
    lines.append(f"Smallest ID→OOD loss: {nice(best[0])} (ΔAUC {100 * best[1]:+.1f})")
print("\n".join("- " + l for l in lines))

In [ ]:
if SAVE_PREFIX:
    table.to_csv(OUT_DIR / f"{SAVE_PREFIX}_table_augmentation.csv")
    table.to_latex(OUT_DIR / f"{SAVE_PREFIX}_table_augmentation.tex", escape=True)
    print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_table_augmentation.csv")